# Awalé — your first agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ml-arena/competition-baseline/blob/main/awale/agent_baseline.ipynb)

Challenge: https://ml-arena.com/challenges/awale

Awalé is a board game for two players: 12 pits, 48 seeds. You will write a
small Python class that chooses a move, test it here, and send it to
ML-Arena, where it plays against the agents of the other participants.

You only need lists, dictionaries, `if`, `for` and one class. Run the cells
from top to bottom.

In [ ]:
import os, sys, subprocess, urllib.request

# The rules of the game, in one small Python file (the platform uses the same one).
if not os.path.exists("awale.py"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/ml-arena/competition-baseline/main/awale/awale.py", "awale.py")
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mlarena-sdk"], check=True)

import awale

## 1. The board

Each player has 6 pits, with 4 seeds in each at the start. When it is your
turn, the platform gives your agent a dictionary called `state`. It always
describes the board **from your side**.

In [ ]:
game = awale.Game()
state = game.state()
state

In [ ]:
awale.show(state)   # the same state, drawn as you would see the table

- `my_pits` are your 6 pits, numbered `0` (your left) to `5` (your right).
- `opponent_pits` are the opponent's 6 pits. They face you, so they are
  printed right to left: `opponent_pits[0]` is just after your pit `5`.
- `legal_moves` lists the pits you are allowed to play.

## 2. One move

To play, you take all the seeds of one of your pits and drop them one by
one in the following pits: first yours, then the opponent's.
`awale.sow` plays a move and returns the board after it.

In [ ]:
my_pits, opponent_pits, captured = awale.sow(state["my_pits"], state["opponent_pits"], 3)
print("my pits      ", my_pits)
print("opponent pits", opponent_pits)
print("captured     ", captured)

Pit `3` is now empty, and its 4 seeds went to my pits `4` and `5`, then to
the opponent's pits `0` and `1`.

**Capturing.** If your *last* seed lands in an opponent's pit and makes
**2 or 3** seeds there, you capture them. The first player with 25 captured
seeds wins. Here the last seed of pit `5` makes 3:

In [ ]:
my_pits, opponent_pits, captured = awale.sow([0, 0, 0, 0, 0, 2], [1, 2, 4, 4, 4, 4], 5)
print("opponent pits", opponent_pits)
print("captured     ", captured)

The last seed made 3 in `opponent_pits[1]`: captured. The pit just before
it, `opponent_pits[0]`, held 2 after the move: captured too. Total 5.

## 3. Your first agent

An agent is a class `Agent` with one method, `choose_move(state)`, which
returns the number of the pit to play. This one plays at random. The first
line saves the cell as the file `agent.py`: that file is your submission.

In [ ]:
%%writefile agent.py
import random


class Agent:
    def choose_move(self, state):
        return random.choice(state["legal_moves"])

## 4. Play here first

`awale.play_match` makes two agents play several games, each moving first
in turn. `awale.RandomAgent` is a ready-made random opponent.

In [ ]:
import importlib
import agent

importlib.reload(agent)        # read agent.py again after each change
awale.play_match(agent.Agent(), awale.RandomAgent(), games=20)

Random against random: about as many wins as losses. To watch one game
move by move:

In [ ]:
game = awale.play_game(agent.Agent(), awale.RandomAgent(), verbose=True)

## 5. Submit

You need your API key: on ml-arena.com, **Profile → API Keys**. In Colab,
open the key icon on the left, add a secret named `MLARENA_API_KEY`, paste
the key and allow this notebook to read it. Never write the key in a cell.

In [ ]:
import mlarena

if "google.colab" in sys.modules:
    from google.colab import userdata
    os.environ["MLARENA_API_KEY"] = userdata.get("MLARENA_API_KEY")
client = mlarena.connect(api_key=os.environ["MLARENA_API_KEY"])

CHALLENGE_ID = 226
client.submit(CHALLENGE_ID, files=["agent.py"], submission_name="random", wait=True)

Your agent now plays matches of 6 games. Open the [challenge page](https://ml-arena.com/challenges/awale):
**Submissions** shows your matches and their replays, **Leaderboard** your
rating.

## 6. A better agent: capture when you can

For each legal move, ask `awale.sow` how many seeds it captures, and keep
the best one. Your agent now uses `awale.py`, so both files are submitted.

In [ ]:
%%writefile agent.py
import random

import awale


class Agent:
    def choose_move(self, state):
        best_moves = []
        best_capture = -1
        for move in state["legal_moves"]:
            my_pits, opponent_pits, captured = awale.sow(
                state["my_pits"], state["opponent_pits"], move)
            if captured > best_capture:
                best_moves = [move]
                best_capture = captured
            elif captured == best_capture:
                best_moves.append(move)
        return random.choice(best_moves)

In [ ]:
importlib.reload(agent)
awale.play_match(agent.Agent(), awale.RandomAgent(), games=100)

In [ ]:
client.submit(CHALLENGE_ID, files=["agent.py", "awale.py"], submission_name="capture", wait=True)

## 7. Your turn

Ideas, from the simplest:

1. **Do not give seeds away.** After your move, the opponent plays. Use
   `awale.sow(opponent_pits, my_pits, reply)` for each of their replies
   (`awale.legal_moves(opponent_pits, my_pits)`) and avoid the moves that
   let them capture a lot.
2. **Protect your pits with 1 or 2 seeds**: they are the ones that can be
   captured.
3. **Look further ahead**: your move, their reply, your next move…

**The clock.** You have 10 seconds for all your moves of one game, and one
move may never take more than 2 seconds. `state["time_left"]` tells you
what is left. If the clock reaches zero you lose that game, so an agent
that thinks a lot should check it. `awale.play_match` uses the same clock
here.

**If something goes wrong.** An exception, or a move that is not in
`legal_moves`, stops the match. The error message is on your submission's
page, with the `state` that caused it: paste that state here and call
`agent.Agent().choose_move(state)` to reproduce it.